In [1]:
# 1. Kết nối Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# !pip install -q einops]

In [ ]:
!cp -r "/content/drive/MyDrive/code KL/One_Peace/onepeace_video_k400.pth" /content
!git clone https://github.com/chien24/One_Peace_FE.git

Cloning into 'One_Peace_FE'...
remote: Enumerating objects: 52, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 52 (delta 26), reused 40 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (52/52), 331.95 KiB | 16.60 MiB/s, done.
Resolving deltas: 100% (26/26), done.


In [ ]:
%cd /content/One_Peace_FE

/content/One_Peace_FE


In [ ]:
!python extract_video_features.py --checkpoint /content/onepeace_video_k400.pth \
    --video_dir "/content/drive/MyDrive/code KL/Omni/data/YouCookII/videos" --output_dir /content/feats \
    --ids_from annotations/youcookii_all.json --stride 8 --batch_size 16 --compile --num_shards 4 --shard_id 0

375 video trong shard 0/4, còn 375 video cần xử lý
Nạp model xong (7s), num_frames=16, dtype=torch.float16
[1/375] --bv0V6ZjWI: 333s ~ 664 clip
    16/664 clip | 0.21 clip/s | còn ~51.6 phút
    576/664 clip | 4.19 clip/s | còn ~0.3 phút
[1/375] --bv0V6ZjWI: XONG (663, 1536) trong 2.5 phút (4.45 clip/s) | TB 4.45 clip/s
[2/375] -GlSSp5ZOCQ: 246s ~ 490 clip
[2/375] -GlSSp5ZOCQ: XONG (490, 1536) trong 0.9 phút (8.66 clip/s) | TB 5.61 clip/s
[3/375] -eyDS81FADw: 147s ~ 292 clip
[3/375] -eyDS81FADw: XONG (292, 1536) trong 0.6 phút (8.28 clip/s) | TB 6.00 clip/s
[4/375] -h8c5sLelZ4: 288s ~ 575 clip
    544/575 clip | 8.59 clip/s | còn ~0.1 phút
[4/375] -h8c5sLelZ4: XONG (575, 1536) trong 1.1 phút (8.46 clip/s) | TB 6.54 clip/s
[5/375] -lMphXmWvbk: 448s ~ 894 clip
    560/894 clip | 8.94 clip/s | còn ~0.6 phút
[5/375] -lMphXmWvbk: XONG (894, 1536) trong 1.7 phút (8.84 clip/s) | TB 7.11 clip/s
[6/375] -yfTO7V2d_E: 580s ~ 1159 clip
    544/1159 clip | 8.86 clip/s | còn ~1.2 phút
    1088/1159 

In [ ]:
!zip -r feats.zip /content/feats

  adding: content/feats/ (stored 0%)
  adding: content/feats/vTE9fobspEw_one_peace_video_finetune.npy (deflated 41%)
  adding: content/feats/0JPDr8uT8xU_one_peace_video_finetune.npy (deflated 41%)
  adding: content/feats/S4At-rC5zNI_one_peace_video_finetune.npy (deflated 41%)
  adding: content/feats/WwbaJPZktec_one_peace_video_finetune.npy (deflated 41%)
  adding: content/feats/YMYNv3cZ9SE_one_peace_video_finetune.npy (deflated 42%)
  adding: content/feats/e-LKNWrv6TQ_one_peace_video_finetune.npy (deflated 41%)
  adding: content/feats/nF67_d1HHnI_one_peace_video_finetune.npy (deflated 41%)
  adding: content/feats/wk0nfwGyPBI_one_peace_video_finetune.npy (deflated 42%)
  adding: content/feats/OJoVVl6PUF0_one_peace_video_finetune.npy (deflated 42%)
  adding: content/feats/nfYzqyureLo_one_peace_video_finetune.npy (deflated 41%)
  adding: content/feats/LpBsoQ6TAL0_one_peace_video_finetune.npy (deflated 42%)
  adding: content/feats/0qbCigxf_sc_one_peace_video_finetune.npy (deflated 41%)
  a

In [ ]:
!ls -lh /content/One_Peace_FE/feats.zip

-rw-r--r-- 1 root root 829M Sep 20 11:46 /content/One_Peace_FE/feats.zip


In [ ]:
!cp -r /content/One_Peace_FE/feats.zip /content/drive/MyDrive

## AUDIO — chạy sau khi xong visual, hoặc ở một phiên Colab riêng

Không chạy visual và audio cùng lúc trong một phiên: hai model không vừa VRAM.

In [ ]:
# A1. Môi trường Python 3.10 riêng cho fairseq cũ của ONE-PEACE (~5 phút)
!git clone -q --depth 1 https://github.com/OFA-Sys/ONE-PEACE /content/ONE-PEACE
!pip install -q uv && uv venv -q --seed --python 3.10 /content/op310
!/content/op310/bin/python -m pip install -q "pip==24.0"
# torch và torchvision phải cài CÙNG một lệnh, nếu không timm sẽ kéo về bản torch CPU
!/content/op310/bin/python -m pip install -q torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu121
# setuptools<81: bản mới bỏ pkg_resources mà librosa 0.10 vẫn import. resampy: cho --res_type kaiser_best
!/content/op310/bin/python -m pip install -q "setuptools<81" "numpy<2" hydra-core==1.0.7 omegaconf==2.0.6     antlr4-python3-runtime==4.8 bitarray sacrebleu tabulate regex timm==0.6.11 iopath tensorboardX pydub     librosa==0.10.0 resampy soundfile soxr einops opencv-python-headless scipy tqdm pillow imageio-ffmpeg
# phải in: torch 2.1.2+cu121 cuda True
!/content/op310/bin/python -c "import torch, librosa, resampy; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
# A2. Copy checkpoint audio ra ổ local (sửa đường dẫn Drive nếu khác)
!cp "/content/drive/MyDrive/code KL/One_Peace/one-peace-audio.pt" /content
!ls -l /content/one-peace-audio.pt   # phải là 6139030001 byte

In [ ]:
# A3. Chạy thật. Bị ngắt thì chạy lại ô này: video đã có .npy sẽ được bỏ qua.
# stride_sec 0.5 = stride 8 của visual. Mặc định: dtype fp16, res_type kaiser_best.
%cd /content/One_Peace_FE
!/content/op310/bin/python extract_audio_features.py --onepeace_repo /content/ONE-PEACE     --checkpoint /content/one-peace-audio.pt     --video_dir "/content/drive/MyDrive/code KL/Omni/data/YouCookII/videos" --output_dir /content/feats_audio     --ids_from annotations/youcookii_all.json --stride_sec 0.5 --batch_size 64

In [ ]:
# A4. Nén và copy về Drive
!zip -r -q /content/feats_audio.zip /content/feats_audio
!ls -lh /content/feats_audio.zip
!cp /content/feats_audio.zip /content/drive/MyDrive

In [ ]:
# A5. (tuỳ chọn) Kiểm tra: gộp visual + audio vào một thư mục rồi so số bước
!mkdir -p /content/feats_all && cp /content/feats/*.npy /content/feats_all/ && cp /content/feats_audio/*.npy /content/feats_all/
!python check_features.py --anno annotations/youcookii_all.json --feat_dir /content/feats_all --stride 8
# Mong đợi: |T_visual - T_audio| = 0-2 bước, chuẩn L2 visual ~20-33, audio ~1.000.
# "thiếu file" sẽ lớn nếu visual mới chạy 1 shard — đó là bình thường.